## Supporting Functions

### Dataset and libraries setup

In [ ]:
#!pip install folktables

In [ ]:
#!git clone https://github.com/Declancharrison/Level-Set-Boosting.git

In [ ]:
#cd Level-Set-Boosting

In [2]:
import sys
sys.path.append('Level-Set-Boosting')

In [3]:
# Relevent libraries
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from folktables import ACSDataSource, ACSEmployment, ACSIncome, ACSTravelTime, ACSPublicCoverage, ACSMobility
from folktables import BasicProblem
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import KFold, cross_val_score
from sklearn.cluster import KMeans
from sklearn.calibration import CalibratedClassifierCV
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from helper_functions import MSCE as MSCE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.utils.validation import check_is_fitted
from sklearn.utils.multiclass import unique_labels
import LSBoost
import helper_functions as hf
from itertools import product
from sklearn.neural_network import MLPClassifier

import seaborn as sns
import cvxpy as cp
from sklearn.naive_bayes import GaussianNB

In [4]:
'''
Functions for implementing multiaccuracy.
'''

'''
This code was produced by the authors of the paper "Universal adaptability: Target-independent inference that competes with propensity scoring"
Paper: https://www.pnas.org/doi/full/10.1073/pnas.2108097119
Code: https://osf.io/kfpr4/?view_only=adf843b070f54bde9f529f910944cd99
'''

import numpy as np
import numpy.ma as ma
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor


class ProbRange:
    """Simple wrapper class for a range of probabilities;
    lower bound is inclusive and upper bound is exclusive. """

    def __init__(self, lower_bound=float('-inf'), upper_bound=float('inf')):
        # We set our lower and upper bounds to -inf and +inf because
        # due to subtleties in how we boost probabilities, we may have
        # negative values or values > 1
        self.lower = lower_bound
        self.upper = upper_bound

    def __eq__(self, other):
        return (isinstance(other, self.__class__) and
                self.lower == other.lower and self.upper == other.upper)

    def __ne__(self, other):
        return (not isinstance(other, self.__class__) or
                self.lower != other.lower or self.upper != other.upper)

    def __hash__(self):
        return (self.lower, self.upper).__hash__()


def within_range_mask(arr, prob_range):
    """
    Given a ProbRange and a numpy array of values, return a boolean mask
    of the values that are within the probability range.
    """
    return (arr >= prob_range.lower) & (arr < prob_range.upper)


def sigmoid(prob, center=False):
    """
    Applies sigmoid function to force probabilities to be between 0 and 1.
    If the probabilities start out being approximately in the range [0, 1],
    then the unaltered sigmoid will output probabilities all greater than 0.5,
    so we recenter the probs at 0.5
    """
    if center:
        prob -= 0.5
    return 1/(1 + np.exp(-1 * prob))


def rescale_prob(prob):
    """
    Rescales probabilities to be between zero and one.
    """
    pred_range = np.max(prob) - np.min(prob)
    new_preds = 1/(pred_range) * prob + (np.min(prob) / pred_range)
    return new_preds


def clip_prob(prob):
    """
    Clips probabilities to be between zero and one.
    """
    prob[prob < 0] = 0
    prob[prob > 1] = 1
    return prob


class RandomLinearPredictor:
    def __init__(self, random_seed=191):
        self.random_seed = random_seed

    def fit(self, data, labels):
        # labels are unused
        np.random.seed(self.random_seed)
        self.coefs = np.random.normal(size=data.shape[1])

    def predict(self, data):
        res = data @ self.coefs.T
        return sigmoid(res)


class ConstantPredictor:
    def __init__(self, constant=0.5):
        self.constant = constant

    def fit(self, data, labels):
        return

    def predict(self, data):
        return np.ones(len(data)) * 0.5


class ResidualFitter:
    def fit_to_resid(self, data, resid):
        try:
            return self.fit(data, resid)
        except Exception as detail:
            print("error:", detail)
        return None

    def fit(self, data, resid):
        raise Exception("Not Implemented")


class RidgeResidualFitter(ResidualFitter):
    def fit(self, data, resid):
        """
        Fits model(s) to predict the residual from the original data. Returns
        the correlation between the predictions and the residual and the model.
        """
        clf = Ridge(alpha=1)
        clf.fit(data, resid)
        h = clf.predict(data)  # TODO: should i use raw labels instead?
        corr = np.mean(h * resid)
        return corr, clf


class TreeResidualFitter(ResidualFitter):
    def fit(self, data, resid):
        """
        Fits trees(s) to predict the residual from the original data. Returns
        the correlation between the predictions and the residual and the model.
        """
        clf = DecisionTreeRegressor(max_depth=2)
        clf.fit(data, resid)
        h = clf.predict(data)
        corr = np.mean(h * resid)
        return corr, clf


class SubgroupModel:
    def __init__(self, subgroup_masks):
        # subgroup masks will allow us to fit the residual by subgroups
        self.subgroup_masks = subgroup_masks
        self.subgroup_preds = {}

    def fit(self, data, resid):
        for s, mask in enumerate(self.subgroup_masks):
            self.subgroup_preds[s] = np.mean(resid[mask])

    def predict(self, data, subgroup_masks=None):
        if subgroup_masks is None:
            for s, m in enumerate(self.subgroup_masks):
                if len(m) != len(data):
                    raise Exception("Length of data is incorrect")
        preds = np.zeros(len(data))
        for s, p in enumerate(self.subgroup_preds):
            preds[self.subgroup_masks[s]] = p
        return preds


class SubgroupFitter(ResidualFitter):
    def __init__(self, subgroup_masks):
        # subgroup masks will allow us to fit the residual by subgroups
        self.subgroup_masks = subgroup_masks

    def fit(self, data, resid):
        """
        Fits model(s) to predict the residual from the original data. Returns
        the correlation between the predictions and the residual and the model.
        """
        m = SubgroupModel(self.subgroup_masks)
        m.fit(data, resid)
        preds = m.predict(data)
        corr = np.mean(preds * resid)
        return corr, m


class SubpopPredictor:
    def __init__(self, subpop, value):
        self.subpop = subpop
        self.value = value

    def predict(self,data):
        return np.array([self.subpop(pt)*self.value for _,pt in data.iterrows()])


class SubpopFitter(ResidualFitter):
    def __init__(self, subpops):
        self.subpops = [(lambda name: lambda pt: pt[name])(attrName) for attrName in subpops]

    def fit(self, data, resid):
        worstCorr = 0
        worstSubpop = lambda pt: 0
        for S in self.subpops:
            sub = np.array([S(pt) for _,pt in data.iterrows()])
            corr = np.mean(sub * resid)
            #print(corr)
            if np.abs(corr) > np.abs(worstCorr):
                worstCorr = corr
                worstSubpop = S
        return abs(worstCorr),SubpopPredictor(worstSubpop,worstCorr)




class MCBoost:
    """
    Wrapper class for the multiaccuracy/multicalibration algorithm.
    """
    def __init__(self,
                 max_iter=5,
                 alpha=1e-4,
                 eta=1,
                 partition=False,
                 num_buckets=2,
                 bucket_strategy="simple",
                 rebucket=False,
                 multiplicative=False,
                 subpop_fitter=None,
                 subpops=None,
                 default_model_class=ConstantPredictor,
                 init_predictor=None
                 ):
        """
        Code to initialize an MCBoost object.
            - max_iter: The maximum number of iterations of the
                multicalibration/multiaccuracy method
            - alpha: accuracy parameter that determines the stopping condition
            - eta: parameter for multiplicative weight update (step size)
            - partition: boolean True/False flag for whether to split up
                predictions by their "partition" (e.g., predictions less than
                0.5 and predictions greater than 0.5)
            - num_buckets: how many buckets to split up the [0, 1]
                probability range
            - bucket_strategy: determines how the buckets will be "explored,"
                currently does nothing
            - rebucket: if True, we perform multicalibration; if false, we
                perform multiaccuracy
            - multiplicative: specifies the strategy for updating the weights
                (multiplicative weight vs additive)
            - subpops:  specifies a collection of characteristic attributes
            	and the values they take defining the S in subpops
                e.g. C = {'age': ['20-29','30-39','40+'], 'nJobs': [0,1,2,'3+'],... etc.}
            - subpop_fitter: specifies the type of model used to fit the
                residual fxn ('TreeResidualFitter' or 'RidgeResidualFitter' (default)).
            - random_seed: Mainly used for the default predictor
            - default_model_class: The class of the model that should be used
                as the MCBoost's default predictor model
            - init_predictor: the initial predictor function to use (i.e., if
                the user has a pretrained model)
        """
        # TODO(matthew): categorical & overlapping subgroups, specify a class
        # of functions over this data where we pass in an object that gives in
        # each of these functions list of functions, each function returns some
        # probability [is_male(), is_female(), is_black(), is_white()]
        # rather than predicting mean, evaluate residual and predict on this
        # subgroup
        self.max_iter = max_iter
        self.alpha = alpha
        self.eta = eta
        self.num_buckets = num_buckets
        self.bucket_strategy = bucket_strategy
        self.rebucket = rebucket
        self.partition = partition
        self.multiplicative = multiplicative
        self.iter_corrs = [0] * max_iter

        if subpops is not None:
            self.subpop_fitter = SubpopFitter(subpops)
      # elif subpop_fitter is not None:
      #     self.subpop_fitter = subpop_fitter() #not implemented
        elif subpop_fitter == 'TreeResidualFitter':
            self.subpop_fitter = TreeResidualFitter()
        elif subpop_fitter == 'RidgeResidualFitter':
            self.subpop_fitter = RidgeResidualFitter()
        else:
            self.subpop_fitter = RidgeResidualFitter()

        if init_predictor is None:
            dm = default_model_class()
            self.predictor = lambda x: dm.predict(x)
        else:
            self.predictor = init_predictor

        # for results of training process
        self.iter_models = []  # models fitted at each step
        self.iter_partitions = []  # keep track of the applicable partitions

    def multicalibrate(self, data, labels):
        """
        Performs multiaccuracy/multicalibration boost algorithm.
        (Multicalibration is achieved by setting "rebucket"=True)

        Given an initial hypothesis (in the form of the predictions
        on validation data), labels on validation data, an auditing
        algorithm, and an accuracy parameter alpha, returns a series of
        trained models that can be used to produce multiaccuracy-boosted
        predictions in combination with the original model.

        Returns a list of models and list of the applicable partitions.

        See paper https://arxiv.org/pdf/1805.12317.pdf (Kim et al. 2018).
        """
        pred_probs = self.predictor(data)
        resid = pred_probs - labels
        buckets = [ProbRange()]  # applies to all datapoints
        if self.partition and self.num_buckets > 1:
            frac = 1 / self.num_buckets
            buckets += [ProbRange(b * frac, (b+1) / frac)
                        for b in range(self.num_buckets)]
            buckets[-1].upper = 1.0  # deal with floating point rounding errors

        new_probs = np.array(pred_probs, copy=True)

        for it in range(self.max_iter):
            corrs = np.zeros(len(buckets))
            models = []

            # fit on various partitions
            probs = new_probs if self.rebucket else pred_probs
            for i, partition in enumerate(buckets):
                mask = within_range_mask(probs, partition)
                data_m = data[mask]
                resid_m = resid[mask]
                corrs[i], model = self.subpop_fitter.fit_to_resid(data_m,
                                                                  resid_m)
                models.append(model)


            if corrs.max() < self.alpha:  # lower than threshold
                for k in range(it, self.max_iter):
                    self.iter_corrs[k] = corrs[int(corrs.argmax())]
                break
            else:
                # update prediction probabilities
                self.iter_corrs[it] = corrs[int(corrs.argmax())]
                max_key = buckets[int(corrs.argmax())]
                prob_mask = within_range_mask(probs, max_key)
                self.iter_models.append(models[int(corrs.argmax())])
                self.iter_partitions.append(max_key)
                new_probs = self.update_probs(new_probs, self.iter_models[-1],
                                              data, mask=prob_mask)
                resid = new_probs - labels  # recalculate residuals

        return

    def update_probs(self, orig_preds, model, x, mask=None, **kwargs):
        """ Apply one multiplicative weight update.

        kwargs are passed to the predict function (check SubgroupFitter) """
        deltas = np.zeros(len(orig_preds))
        if mask is not None:  # only update the relevant probabilities
            deltas[mask] = model.predict(x[mask])
        else:
            deltas = model.predict(x, **kwargs)  # TODO: no kwargs hack
        # TODO: consider changing sigmoid to tanh function
        if self.multiplicative:
            update_weights = np.exp(-1 * self.eta * deltas)
            new_preds = update_weights * orig_preds  # sigmoid
        else:
            new_preds = orig_preds + deltas
        # new_preds = rescale_prob(new_preds)
        # new_preds = sigmoid(new_preds)
        new_preds = clip_prob(new_preds)
        return new_preds

    def predict_prob(self, x, t=float('inf'), **kwargs):
        # change method name to predict?
        # able to access predictions at various iterations
        """ Apply multiplicative weight updates using multiple models. Option
        to pass kwargs to the predict function via the update_probs function.
        """
        orig_preds = self.predictor(x)
        new_preds = np.array(orig_preds, copy=True)
        for i, m in enumerate(self.iter_models):
            if i <= t:
                probs = new_preds if self.rebucket else orig_preds
                mask = within_range_mask(probs, self.iter_partitions[i])
                new_preds = self.update_probs(new_preds, m, x, mask=mask,
                                              **kwargs)

        return new_preds

    def predict_all_prob(self, x, t=float('inf'), **kwargs):
        #return for all iterations up to t
        orig_preds = self.predictor(x)
        new_preds = np.array(orig_preds, copy=True)
        all_preds = [np.copy(new_preds)]
        for i, m in enumerate(self.iter_models):
            if i <= t:
                probs = new_preds if self.rebucket else orig_preds
                mask = within_range_mask(probs, self.iter_partitions[i])
                new_preds = self.update_probs(new_preds, m, x, mask=mask,
                                              **kwargs)
                all_preds.append(new_preds)
        for i in range(t - len(all_preds)):
            all_preds.append(new_preds)
        return all_preds



In [ ]:
def gen_preds(model):
    return lambda x: model.predict_proba(x)[:, 1] # Predict function for MCBoost

### Witness Functions

In [5]:
class MAccWitness(BaseEstimator, RegressorMixin):

    def __init__(self, gamma=1, degree = 1, coef0 = 0, metric='rbf'):
        """
        error regression using kernel
        metric: 'rbf', 'linear', 'poly', 'sigmoid'
        rbf parameters: gamma
        poly parameters: degree, coef0, gamma
        sigmoid parameters: coef0, gamma
        linear parameters: None
        """
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.metric = metric

    def fit(self, X, y=None):
        """
        prepare data for regression of error
        using all training samples to define witness function
        """
        self.X_ = X
        self.y_ = y.reshape(len(y),1)
        self.n_ = len(self.y_)

        return self
    def predict(self, X, y=None):
        """
        making prediction of error on new data (y is error here)
        X: N * 512
        X_: M * 512
        K: N * M
        y_: M * 1
        prod: N * 1d
        """
        # compute the cos similarity between X and training samples
        if self.metric == 'rbf':
            K = pairwise_kernels(X,self.X_,metric=self.metric,gamma=self.gamma)
        elif self.metric == 'linear':
            K = pairwise_kernels(X,self.X_,metric=self.metric)
        elif self.metric == 'poly':
            K = pairwise_kernels(X,self.X_,metric=self.metric, degree=self.degree, coef0=self.coef0, gamma=self.gamma)
        elif self.metric == 'sigmoid':
            K = pairwise_kernels(X,self.X_,metric=self.metric, coef0=self.coef0, gamma=self.gamma)
        else:
            raise ValueError('metric not supported')
        prod = np.ravel(K@self.y_)

        return prod

    def score(self, X, y=None):
        # counts number of values bigger than mean
        return(np.abs(np.corrcoef(self.predict(X),y)[0,1]))


### Achieving calibration with witness function

In [6]:
class KMultiAcc(BaseEstimator, RegressorMixin):
  def __init__(self, baseline_model="Logistic_Regression"):
    # baseline classifiers
    if baseline_model == "Logistic_Regression":
      self.model = LogisticRegression()
    elif baseline_model == "Decision_Tree":
      self.model = DecisionTreeClassifier(max_depth = 1)
    elif baseline_model == "Random_Forest":
      self.model = RandomForestClassifier(max_depth=2, random_state=0)
    elif baseline_model == "Kernel_SVM":
      self.model = SVC(gamma="auto", probability = True)
    elif baseline_model == "Naive_Bayes":
      self.model = GaussianNB()
    elif baseline_model == "NN":
      self.model = MLPClassifier(random_state=1, max_iter=300)
    else:
      return "Baseline not supported"

    self.wit_model = None
    self.opt_lambda_ = None
    self.baseline_model = baseline_model
    self._is_fitted = False
    self.wit_model_ = None

  def fit_model(self, X, y):
    self.model.fit(X, y)

  def fit(self, X_train_wit_val, y_wit_train_val, X_wit = None, y_wit = None, X_val = None, y_val = None, witness_metric = 'rbf', alpha = .01):
    if X_val is None and X_wit is None:
      X_train, X_wit_val, y_train, y_wit_val = train_test_split(X_train_wit_val, y_wit_train_val, test_size=0.5) # Use test_size = .5 for non-qp
      X_wit, X_val, y_wit, y_val = train_test_split(X_wit_val, y_wit_val, test_size=0.5) # Use test_size = .5 for non-qp
      self.fit_model(X_train, y_train)

    self.classes_ = unique_labels(y_wit)

    yhat_proba_wit = self.model.predict_proba(X_wit)[:, 1]
    error_wit = y_wit - yhat_proba_wit

    yhat_proba_val = self.model.predict_proba(X_val)[:, 1]
    error_val = y_val - yhat_proba_val

    # search for optimal rbf kernel param using witness set
    gopt = grid_search_params(witness_metric, X_wit, error_wit)

    # define witness on train and validation
    wit = MAccWitness(gamma=gopt, metric=witness_metric)

    self.wit_model = make_pipeline(StandardScaler(), wit)
    self.wit_model.fit(X_wit, error_wit)

    # search for best lambda (parameter for the updated predictor)
    wit_val = self.wit_model.predict(X_val)

    ## Put in QP
    """opt_l, eps = self.solve_qp(yhat_proba_val, y_val, wit_val)
    self.lambda_opt = opt_l
    print(min(yhat_proba_val + opt_l * wit_val), max(yhat_proba_val + opt_l * wit_val))
    print(f"Lambda: {opt_l}")
    print(f"{np.sum(eps), max(eps), min(eps)}")"""
    lambdas = np.arange(0, .1, 0.003)
    divergence = np.zeros(len(lambdas))
    calibration_error = np.zeros(len(lambdas))

    self.lambda_opt = None
    for i in range(len(lambdas)):
      g_val, g_val_pred = self.update_proba(yhat_proba_val, lambdas[i], wit_val)
      divergence[i] = np.linalg.norm(yhat_proba_val - g_val)
      calibration_error[i] = compute_calibration_error(wit_val, y_val, g_val)
    for i in np.argsort(divergence):
      if calibration_error[i] < alpha:
        self.lambda_opt = lambdas[i]
        break
    if self.lambda_opt is None: self.lambda_opt = lambdas[np.nanargmin(calibration_error)]

    print(f"Optimal lambda: {self.lambda_opt}")


    self._is_fitted = True
    self.wit_model_ = self.wit_model

    return self

  def predict_proba(self,X):
    if self.wit_model == None:
      raise ValueError("No Witness was fit!")

    yhat_proba_test = self.model.predict_proba(X)[:,1]
    wit_test = self.wit_model.predict(X)
    yhat_proba_updated, yhat_updated = self.update_proba(yhat_proba_test, self.lambda_opt, wit_test)
    return np.column_stack((1 - yhat_proba_updated, yhat_proba_updated))

  def predict(self, X):
    if self.wit_model == None:
      raise ValueError("No Witness was fit!")
    yhat_proba_test = self.model.predict_proba(X)[:,1]
    wit_test = self.wit_model.predict(X)
    yhat_proba_updated, yhat_updated = self.update_proba(yhat_proba_test, self.lambda_opt, wit_test)
    return yhat_updated

  # updated model algorithm using witness
  def update_proba(self, yhat_proba, lambda_, wit_value):
    yhat_proba = yhat_proba + lambda_ * wit_value
    # set all <0 values to 0, all >1 values to 1
    yhat_proba[yhat_proba < 0] = 0
    yhat_proba[yhat_proba > 1] = 1
    yhat_pred = (yhat_proba > 0.5).astype('int')
    return yhat_proba, yhat_pred

  def solve_qp(self, f, y, c, alpha = .02):
    #f is predictor on validation points
    #y is true label validation points
    #c is witness function applied to validation points

    with open('qp.npy', 'wb') as file:
      np.save(file, f)
      np.save(file, y)
      np.save(file, c)

    n = len(f) #n is dim

    #alpha is multiaccuracy constraint

    f.reshape((len(f), 1))

    A = np.row_stack((c.T / n, -1 * c.T / n, np.diag(np.ones(n)), np.diag(np.ones(n))))
    print(f"A: {A.shape}")
    b = np.row_stack((alpha + c.T @ y / n, alpha - c.T @ y / n, np.ones((n, 1)), np.ones((n, 1))))
    print(f"b: {b.shape}")
    print(f"n: {n}")

    print(f"A@f: {(A @ f / n).shape}")
    Bmat = 1 / 2 * A @ A.T
    d = b - (A @ f).reshape((len(b), 1))

    print(f"Bmat: {Bmat.shape}")
    #print(f"Rank of Bmat: {np.linalg.matrix_rank(Bmat)}")
    """u, s, v = np.linalg.svd(A)
    print(f"SVD: {s}")
    plt.hist(s)
    plt.show()
    print("f")
    print(f"Condition number of Bmat: {np.linalg.cond(Bmat)}")
    eigdecomp = np.linalg.eig(Bmat)
    print(f"Notable eig of Bmat: {eigdecomp[0]}")
    print(f"Norm of C: {np.linalg.norm(c)}")
    plt.hist(c)
    plt.show()"""
    print(f"d: {d.shape}")
    print(f"f: {f.shape}")

    L = cp.Variable((2 * n + 2, 1))
    print(f"L: {L.shape}")
    constraints = [0 <= L]
    objective = cp.Minimize(cp.quad_form(L, cp.Parameter(shape=Bmat.shape, value = Bmat, PSD=True)) + d.T @ L)
    prob = cp.Problem(objective, constraints)

    result = prob.solve(solver=cp.SCS)
    val = L.value
    l = (val[1] - val[0]) / n
    print(f"Lambda dual: {val[0], val[1]}")
    eps = (val[n+2:] - val[2:n+2])
    return l, eps

In [7]:
# updated model algorithm using witness
def update_proba(yhat_proba, lambda_, wit_value):
  yhat_proba = yhat_proba + lambda_ * wit_value
  # set all <0 values to 0, all >1 values to 1
  yhat_proba[yhat_proba < 0] = 0
  yhat_proba[yhat_proba > 1] = 1
  yhat_pred = (yhat_proba > 0.5).astype('int')
  return yhat_proba, yhat_pred

In [8]:
# compute calibration error
def compute_calibration_error(wit_value, y_pred, yhat_proba):
  return np.abs((wit_value * (y_pred - yhat_proba)).mean())

# conditioned calibration error
def con_cal_err(bin, wit_value, y_pred, yhat_proba):
  '''
  bin is the number of slices
  wit_value is c*(x)
  y_pred is the predicted label (0 or 1)
  yhat_proba is the probability score (between 0 and 1)
  '''
  err = 0
  incre = 1/bin
  for i in range(bin):
    ind = np.where(np.logical_and(yhat_proba>i*incre, yhat_proba<=(i+1)*incre))
    temp_sum = compute_calibration_error(wit_value[ind], y_pred[ind], yhat_proba[ind])
    # print(temp_sum)
    if (not np.isnan(temp_sum)):
      frac = len(ind[0]) / len(yhat_proba)
      err += (temp_sum ** 2) * frac
  return err

In [9]:
# Mega Function that takes in dataset
def achieving_calibration_with_witness(features, label, baseline_model="Logistic_Regression", witness_metric = 'rbf'):
  scaler = StandardScaler()
  #fit transform data
  features = scaler.fit_transform(features, label)
  label = label.astype('int')

  base_msce = []
  base_KCE = []
  kmulcal_msce = []
  kmulcal_KCE = []
  lsboost_msce = []
  lsboost_KCE = []
  base_msce_binned = []
  base_binned_KCE = []
  kmulcal_msce_binned = []
  kmulcal_binned_KCE = []
  MCBoost_msce = []
  MCBoost_KCE = []
  isotonic_msce = []
  isotonic_KCE = []
  sigmoid_msce = []
  sigmoid_KCE = []
  ablated_msce = []
  ablated_KCE = []
  AUC = []

  save_np = []
  for seed in range(2, 25, 5):
    #try:
    X_train_wit_val, X_test, y_train_wit_val, y_test = train_test_split(
        features, label, test_size=0.3, random_state=seed)

    X_train, X_wit_val, y_train, y_wit_val = train_test_split(
        X_train_wit_val, y_train_wit_val, test_size=0.4, random_state=seed)

    X_wit, X_val, y_wit, y_val = train_test_split(X_wit_val, y_wit_val, test_size=0.4, random_state=seed)

    if baseline_model == "Logistic_Regression":
      model = LogisticRegression()
      weak_learner = LogisticRegression()
    elif baseline_model == "Decision_Tree":
      model = DecisionTreeClassifier(max_depth = 1)
      weak_learner = DecisionTreeClassifier(max_depth = 1)
    elif baseline_model == "Random_Forest":
      model = RandomForestClassifier(max_depth=2, random_state=seed)
      weak_learner = RandomForestClassifier(max_depth=2, random_state=seed)
    elif baseline_model == "Kernel_SVM":
      model = SVC(gamma="auto", probability = True)
      weak_learner = SVC(gamma="auto", probability = True)
    elif baseline_model == "Naive_Bayes":
      model = GaussianNB()
      weak_learner = GaussianNB()
    elif baseline_model == "NN":
      model = MLPClassifier(random_state=seed, max_iter=300)
      weak_learner = MLPClassifier(random_state=1, max_iter=300)
    else:
      return "Baseline not supported"


    kma = KMultiAcc(baseline_model = baseline_model)
    kma.fit_model(X_train, y_train)
    kma.fit(X_wit_val, y_wit_val, X_wit, y_wit, X_val, y_val)

    yhat_proba_test = kma.model.predict_proba(X_test)[:,1]
    wit_test = kma.wit_model.predict(X_test)
    g_test, g_test_pred = update_proba(yhat_proba_test, kma.lambda_opt, wit_test)

    # validation dt
    wit_val = kma.wit_model.predict(X_val)
    yhat_proba_val = kma.model.predict_proba(X_val)[:,1]
    g_val, g_val_pred = update_proba(yhat_proba_val, kma.lambda_opt, wit_val)

    # Ours w isotonic calibration
    model_isotonic = CalibratedClassifierCV(kma, cv="prefit", method="isotonic")
    model_isotonic.fit(X_val, y_val)
    prob_pos_isotonic = model_isotonic.predict_proba(X_test)[:, 1]

    # Ours w sigmoid calibration
    model_sigmoid = CalibratedClassifierCV(kma, cv="prefit", method="sigmoid")
    model_sigmoid.fit(X_val, y_val)
    prob_pos_sigmoid = model_sigmoid.predict_proba(X_test)[:, 1]

    #Ablated Isotonic Calibration
    model.fit(X_train, y_train)
    ablated_isotonic = CalibratedClassifierCV(model, cv="prefit", method="isotonic")
    ablated_isotonic.fit(X_val, y_val)
    prob_pos_ablated = ablated_isotonic.predict_proba(X_test)[:, 1]


    #LSBoost
    LSBoostReg = LSBoost.LSBoostingRegressor(
                                  T = 100,
                                  num_bins = 100,
                                  min_group_size = 5,
                                  global_gamma = .005,
                                  weak_learner=weak_learner,
                                  bin_type = 'distribution',
                                  learning_rate = .1,
                                  initial_model = None,
                                  final_round = True,
                                  center_mean=False)
    LSBoostReg.fit(X_train_wit_val, y_train_wit_val)

    training_predictions = LSBoostReg.predict(X_train)
    test_predictions = LSBoostReg.predict(X_test)

    # MCBoost
    model.fit(X_train, y_train)
    mcrf1 = MCBoost(partition = True, multiplicative = True, init_predictor = gen_preds(model), max_iter = 10)
    mcrf1.multicalibrate(X_train_wit_val, y_train_wit_val)

    MCBoost_test = mcrf1.predict_prob(X_test)


    base_msce_temp, base_KCE_temp, kmulcal_msce_temp, kmulcal_KCE_temp, lsboost_msce_temp, \
    lsboost_KCE_temp, base_msce_binned_temp, base_binned_KCE_temp, kmulcal_msce_binned_temp, kmulcal_binned_KCE_temp, \
    MCBoost_msce_temp, MCBoost_KCE_temp, isotonic_cal_msce_temp, isotonic_cal_kce_temp, \
    sigmoid_cal_msce_temp, sigmoid_cal_kce_temp, ablated_msce_temp, ablated_kce_temp, AUC_temp \
                              = mega_compute(wit_test, y_test, yhat_proba_test, test_predictions, \
                                              g_test, MCBoost_test, baseline_model, prob_pos_isotonic, \
                                              prob_pos_sigmoid, prob_pos_ablated)

    base_msce.append(base_msce_temp)
    base_KCE.append(base_KCE_temp)
    kmulcal_msce.append(kmulcal_msce_temp)
    kmulcal_KCE.append(kmulcal_KCE_temp)
    lsboost_msce.append(lsboost_msce_temp)
    lsboost_KCE.append(lsboost_KCE_temp)
    base_msce_binned.append(base_msce_binned_temp)
    base_binned_KCE.append(base_binned_KCE_temp)
    kmulcal_msce_binned.append(kmulcal_msce_binned_temp)
    kmulcal_binned_KCE.append(kmulcal_binned_KCE_temp)
    MCBoost_msce.append(MCBoost_msce_temp)
    MCBoost_KCE.append(MCBoost_KCE_temp)
    isotonic_msce.append(isotonic_cal_msce_temp)
    isotonic_KCE.append(isotonic_cal_kce_temp)
    sigmoid_msce.append(sigmoid_cal_msce_temp)
    sigmoid_KCE.append(sigmoid_cal_kce_temp)
    ablated_msce.append(ablated_msce_temp)
    ablated_KCE.append(ablated_kce_temp)
    AUC.append(AUC_temp)

    save_np += [wit_test, y_test, yhat_proba_test, test_predictions, g_test, \
                MCBoost_test, prob_pos_isotonic, prob_pos_ablated, \
                base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, \
                kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, \
                isotonic_msce, isotonic_KCE, ablated_msce, ablated_KCE, AUC]
    #except:
    #  print("Error occured on this run.")

  #save_np = np.array(save_np)
  #save_df = pd.DataFrame(save_np)
  #csv_file_path = baseline_model + '.csv'
  #save_df.to_csv(csv_file_path)

  return base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, \
         base_msce_binned, base_binned_KCE, kmulcal_msce_binned, kmulcal_binned_KCE, \
         MCBoost_msce, MCBoost_KCE, isotonic_msce, isotonic_KCE, sigmoid_msce, sigmoid_KCE, \
         ablated_msce, ablated_KCE, AUC

### Helper Functions

In [10]:
def mega_compute(wit_test, y_test, yhat_proba_test, test_predictions, g_test, MCBoost_test, baseline_model, prob_pos_isotonic, prob_pos_sigmoid, prob_pos_ablated):

  if baseline_model == "Logistic_Regression":
    num_cluster = 5
  elif baseline_model == "Decision_Tree":
    num_cluster = 7
  elif baseline_model == "Random_Forest":
    num_cluster = 5
  else:
    num_cluster = 3

  g_test_binned = KMeans(n_clusters=num_cluster, random_state=0, n_init="auto").fit(g_test.reshape(-1, 1))

  yhat_proba_test_binned = KMeans(n_clusters=num_cluster, random_state=0, n_init="auto").fit(yhat_proba_test.reshape(-1, 1))

  g_new = np.zeros(len(g_test))
  yhat_new = np.zeros(len(yhat_proba_test))
  for i in range(num_cluster): # number of cluster here
    ind1 = np.where(g_test_binned.labels_ ==i )
    ind2 = np.where(yhat_proba_test_binned.labels_ == i)
    g_new[ind1] = g_test_binned.cluster_centers_[i]
    yhat_new[ind2] = yhat_proba_test_binned.cluster_centers_[i]

  # print(g_test_binned_new)

  # compare c* error_test and c* g_error_test
  base_KCE = compute_calibration_error(wit_test, y_test, yhat_proba_test)
  print(f"Baseline Kernel calibration error: {base_KCE}")

  kmulcal_KCE = compute_calibration_error(wit_test, y_test, g_test)
  print(f"Our Method's kernel calibration error: {kmulcal_KCE}")

  lsboost_KCE = compute_calibration_error(wit_test,y_test, test_predictions)
  print(f"LSBoost kernel calibration error: {lsboost_KCE}")

  # kernel error for binned f(x) and g(x)
  base_binned_KCE = compute_calibration_error(wit_test, y_test, yhat_new)
  print(f"Baseline Kernel calibration error, binned: {base_binned_KCE}")

  kmulcal_binned_KCE = compute_calibration_error(wit_test, y_test, g_new)
  print(f"Our Method's Kernel calibration error, binned: {kmulcal_binned_KCE}")

  # # condition calibration error
  # con_cal_error1 = con_cal_err(num_bins, wit_test, y_test, g_test)
  # print(f"our condition calibration error: {con_cal_error1}")

  # con_cal_error2 = con_cal_err(num_bins, wit_test, y_test, test_predictions)
  # print(f"BoostReg condition calibration error: {con_cal_error2}")

  # compare standard calibration metric
  base_msce = MSCE(y_test, yhat_proba_test)
  print(f'Baseline MSCE: {base_msce:.6f}')

  kmulcal_msce = MSCE(y_test, g_test)
  print(f'Our Method\'s MSCE: {kmulcal_msce:.6f}')

  lsboost_msce = MSCE(y_test, test_predictions)
  # print(f'Our Method\'s MS Calibration Error: {MSCE(y_test, g_test):.6f}')
  print(f'LSBoost MSCE: {lsboost_msce:.6f}')

  # MSCE for binned f(x) and g(x)
  base_msce_binned = MSCE(y_test, yhat_new)
  print(f'Baseline MSCE, binned: {base_msce_binned:.6f}')

  kmulcal_msce_binned = MSCE(y_test, g_new)
  print(f'Our Method\'s MSCE, binned: {kmulcal_msce_binned:.6f}')

  MCBoost_msce = MSCE(y_test, MCBoost_test)
  MCBoost_KCE = compute_calibration_error(wit_test, y_test, MCBoost_test)

  isotonic_cal_msce = MSCE(y_test, prob_pos_isotonic)
  isotonic_cal_KCE = compute_calibration_error(wit_test, y_test, prob_pos_isotonic)

  sigmoid_cal_msce = MSCE(y_test, prob_pos_sigmoid)
  sigmoid_cal_KCE = compute_calibration_error(wit_test, y_test, prob_pos_sigmoid)

  ablated_msce = MSCE(y_test, prob_pos_ablated)
  ablated_KCE = compute_calibration_error(wit_test, y_test, prob_pos_ablated)

  import matplotlib.pyplot as plt
  plt.hist(g_new, label='ours')
  plt.hist(test_predictions)
  plt.title(f'K Means data visualization for {baseline_model}')
  plt.show()

  # auc = roc_auc_score(y_test, )
  AUC = [roc_auc_score(y_test, yhat_proba_test), roc_auc_score(y_test, g_test), \
         roc_auc_score(y_test, test_predictions), \
         roc_auc_score(y_test, g_new), roc_auc_score(y_test, MCBoost_test), \
         roc_auc_score(y_test, prob_pos_isotonic), \
         roc_auc_score(y_test, prob_pos_ablated)] #,


  return base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, \
         base_msce_binned, base_binned_KCE, kmulcal_msce_binned, kmulcal_binned_KCE, \
         MCBoost_msce, MCBoost_KCE, isotonic_cal_msce, isotonic_cal_KCE, sigmoid_cal_msce,\
         sigmoid_cal_KCE, ablated_msce, ablated_KCE, AUC


In [11]:
def plotter(base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC, baseline_model, prefix = ""):
  x = np.array([np.mean(base_KCE, axis = 0), np.mean(kmulcal_KCE, axis = 0), np.mean(lsboost_KCE, axis = 0), \
                np.mean(kmulcal_binned_KCE, axis = 0), np.mean(MCBoost_KCE, axis = 0), np.mean(isotonic_KCE, axis = 0), np.mean(ablated_KCE, axis = 0)])
  y = np.array([np.mean(base_msce, axis = 0), np.mean(kmulcal_msce, axis = 0), np.mean(lsboost_msce, axis = 0), \
                np.mean(kmulcal_msce_binned, axis = 0), np.mean(MCBoost_msce, axis=0), np.mean(isotonic_msce, axis = 0), np.mean(ablated_msce, axis = 0)])
  z = np.mean(AUC, axis = 0) ## replace with AUC values
  # l = ['Baseline', 'KMultiCal', 'LS Boost', 'Baseline + KMeans', 'KMultiCal + KMeans']

  # markers = ['o', 's', '^', 'D', 'v']

  x_std = np.array([np.std(base_KCE, axis = 0), np.std(kmulcal_KCE, axis = 0), np.std(lsboost_KCE, axis = 0), np.std(kmulcal_binned_KCE, axis = 0), np.std(MCBoost_KCE, axis = 0), np.std(isotonic_KCE, axis = 0), np.std(ablated_KCE, axis = 0)])

  y_std = np.array([np.std(base_msce, axis = 0), np.std(kmulcal_msce, axis = 0), np.std(lsboost_msce, axis = 0),  np.std(kmulcal_msce_binned, axis = 0), np.std(MCBoost_msce, axis=0), np.std(isotonic_msce, axis = 0), np.std(ablated_msce, axis = 0)])


  colors = ['steelblue','darkgoldenrod','coral','gray','limegreen', 'darkred', 'indigo']
  markersize = 40
  sns.set(style="whitegrid", color_codes=True)
  base = plt.scatter(x[0], y[0], c=colors[0], marker = 'o', s = markersize)
  kma = plt.scatter(x[1], y[1], c=colors[1], marker = 's', s = markersize)
  lsb = plt.scatter(x[2], y[2], c=colors[2], marker = 'p', s = markersize)
  kmak = plt.scatter(x[3], y[3], c=colors[3], marker = 'x', s = markersize)
  mcb = plt.scatter(x[4], y[4], c=colors[4], marker = '^', s = markersize)
  iso = plt.scatter(x[5], y[5], c=colors[5], marker = 'd', s = markersize)
  abl = plt.scatter(x[6], y[6], c=colors[6], marker = 'h', s = markersize)

  norm = plt.Normalize(0.7, 1)
  m = plt.cm.ScalarMappable(cmap="Reds", norm = norm)
  m.set_array([])

  # cm=plt.get_cmap('Reds')
  # for i, (xval, yval, x_error_val, y_error_val, zval) in enumerate(zip(x, y, x_std, y_std, z)):
  #     # colour=cm(0.7*zval)
  #     # print(zval)
  #     plt.errorbar(xval, yval, xerr=x_error_val, yerr=y_error_val, linestyle='', ecolor=cm(zval), alpha = 0.3, capsize = 3)

  plt.errorbar(x[0], y[0], xerr=x_std[0], yerr=y_std[0], linestyle='', color = colors[0], alpha = 1)
  plt.errorbar(x[1], y[1], xerr=x_std[1], yerr=y_std[1], linestyle='', color= colors[1], alpha = 1)
  plt.errorbar(x[2], y[2], xerr=x_std[2], yerr=y_std[2], linestyle='', color= colors[2], alpha = 1)
  plt.errorbar(x[3], y[3], xerr=x_std[3], yerr=y_std[3], linestyle='', color= colors[3], alpha = 1)
  plt.errorbar(x[4], y[4], xerr=x_std[4], yerr=y_std[4], linestyle='', color= colors[4], alpha = 1)
  plt.errorbar(x[5], y[5], xerr=x_std[5], yerr=y_std[5], linestyle='', color= colors[5], alpha = 1)
  plt.errorbar(x[6], y[6], xerr=x_std[6], yerr=y_std[6], linestyle='', color= colors[6], alpha = 1)

  # base.figure.colorbar(m, label = 'AUC')
  # cbar = plt.colorbar(base)
  # cbar.set_label('AUC')

  plt.xlabel('KME', weight='bold')
  plt.ylabel('MSCE', weight='bold')

  if baseline_model == "Logistic_Regression":
    plt.title('Logistic Regression', weight='bold')
  elif baseline_model == "Decision_Tree":
    plt.title('Decision Tree', weight='bold')
  elif baseline_model == "Random_Forest":
    plt.title('Random Forest', weight='bold')
  elif baseline_model == "Kernel_SVM":
    plt.title('Kernel SVM', weight='bold')
  elif baseline_model == "Naive_Bayes":
    plt.title("Gaussian Naive Bayes", weight='bold')
  elif baseline_model == "NN":
    plt.title("Neural Network", weight='bold')
  else:
    plt.title('')

  plt.rcParams["font.family"] = "monospace"
  plt.rcParams['font.size']=15

  plt.ylim(bottom=-.01)
  plt.xlim(left=0)

  #### legend only for last plot
  plt.legend((base, kma, lsb, kmak, mcb, iso, abl),
           (f'baseline (AUC: {z[0]:.3f})', f'KMultiAcc (AUC: {z[1]:.3f})', \
            f'LS Boost (AUC: {z[2]:.3f})', f'KMultiAcc + KMeans (AUC: {z[3]:.3f})', f'MC Boost (AUC: {z[4]:.3f})', \
            f'KMultiAcc + Isotonic Calibration (AUC: {z[5]:.3f})', f'Baseline + Isotonic Calibration (AUC: {z[6]:.3f})'))

  plt.savefig('plots/'+ prefix + baseline_model + '.png',format='png', dpi=300)
  plt.show()


In [12]:
# grid search on the best parameters
def grid_search_params(witness_metric, X_val, y_val):
    n_splits = 5
    kf = KFold(n_splits=n_splits, random_state = 42, shuffle=True)
    if witness_metric == 'rbf':
        gamma = np.arange(1, 10, 0.5)
        scores = np.zeros(len(gamma))
        for g in range(len(gamma)):
            wit = MAccWitness(gamma=gamma[g], metric=witness_metric)
            wit_model = make_pipeline(StandardScaler(), wit)
            score = cross_val_score(wit_model,X_val ,y_val,cv=kf,n_jobs=n_splits).mean()
            scores[g] = score
        #defining the optimal found witness function
        idx_max = np.nanargmax(scores.flatten())
        gopt = gamma[idx_max]
        print(f"Optimal Gamma: {gopt}")
        return gopt

    elif witness_metric == 'sigmoid':
        gamma = np.arange(1,10,0.5)
        coef0 = np.arange(-5, 5, 1)
        combos = list(product(range(len(gamma)), range(len(coef0))))
        scores = np.zeros((len(gamma), len(coef0)))
        #evaluating for each gamma the resulting witness function
        for g, c in combos:
            wit = MAccWitness(gamma=gamma[g], metric=witness_metric, coef0=coef0[c])
            wit_model = make_pipeline(StandardScaler(), wit)
            score = cross_val_score(wit_model,X_val,y_val,cv=kf,n_jobs=n_splits).mean()
            scores[g, c] = score
        #defining the optimal found witness function
        idx_max = np.nanargmax(scores.flatten())
        gopt, copt = combos[idx_max]
        gopt, copt = gamma[gopt], coef0[copt]
        print(f"Optimal Gamma: {gopt}, Optimal Coef: {copt}")
        return [gopt, copt]

    elif witness_metric == 'poly':
        # Optimal Gamma: 0.05, Optimal Coef: 1.5, Optimal Degree: 2
        gamma = [0.04, 0.05, 0.06, 0.1, 0.5]
        coef0 = [0, 0.5, 0.7, 1, 1.5, 2]
        degree = [2,3,4]
        combos = list(product(range(len(gamma)), range(len(coef0)), range(len(degree))))
        scores = np.zeros((len(gamma), len(coef0), len(degree)))

        #evaluating for each gamma the resulting witness function
        for g, c, d in combos:
            wit = MAccWitness(gamma=gamma[g], metric=witness_metric, coef0=coef0[c], degree=degree[d])
            wit_model = make_pipeline(StandardScaler(), wit)
            score = cross_val_score(wit_model,X_val,y_val,cv=kf,n_jobs=n_splits).mean()
            scores[g, c] = score
        #defining the optimal found witness function
        idx_max = np.nanargmax(scores.flatten())
        gopt, copt, dopt = combos[idx_max]
        gopt, copt, dopt = gamma[gopt], coef0[copt], degree[dopt]
        print(f"Optimal Gamma: {gopt}, Optimal Coef: {copt}, Optimal Degree: {dopt}")
        return [gopt, copt, dopt]

In [13]:
def plot_for_task(features, labels, base_classifiers, prefix):
  for classifier in base_classifiers:
    base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
    kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, \
    isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC = achieving_calibration_with_witness(features, labels, classifier)

    plotter(base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
    kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, \
    isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC, classifier, prefix)

## Experiments

### Folktables Experiments

In [14]:
!mkdir plots/

In [ ]:
base_classifiers = ["NN", "Logistic_Regression", "Random_Forest", "Naive_Bayes"]
#base_classifiers = ["Random_Forest", "Naive_Bayes", "NN"]

In [ ]:
# Employment task Al Short
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=["AL"], download=True)
features, labels, group = ACSEmployment.df_to_numpy(acs_data)
prefix = "EMP_AL_"

print(len(features))

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# Employment task MA
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=["MA"], download=True)
features, labels, group = ACSEmployment.df_to_numpy(acs_data)
prefix = "EMP_MA_"

print(len(features))

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# Income Task WA
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["WA"], download=True)
features, labels, _ = ACSIncome.df_to_numpy(data)
prefix = "Income_WA_"

print(len(features))

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# Income Task IL
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["IL"], download=True)
features, labels, _ = ACSIncome.df_to_numpy(data)
prefix = "Income_IL_"

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# Health Public Coverage Task OH
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["OH"], download=True)
features, labels, _ = ACSPublicCoverage.df_to_numpy(data)
prefix = "Health_OH_"

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# Health Public Coverage Task WI
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["WI"], download=True)
features, labels, _ = ACSPublicCoverage.df_to_numpy(data)
prefix = "Health_WI_"

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# ACS Mobility Task NJ
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["NJ"], download=True)
features, labels, _ = ACSMobility.df_to_numpy(data)
prefix = "Mobility_NJ_"

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# ACS Mobility Task NY
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["NY"], download=True)
features, labels, _ = ACSMobility.df_to_numpy(data)
prefix = "Mobility_NY_"

plot_for_task(features, labels, base_classifiers, prefix)

In [ ]:
# Income Task WA
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["WA"], download=True)
features, labels, _ = ACSIncome.df_to_numpy(data)
prefix = "Income_WA_"

base_classifiers = ["Logistic_Regression", "Naive_Bayes", "Random_Forest", "NN"]

data_class = []

for classifier in base_classifiers:
    base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
    kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce_Emp_AL_RF, \
    isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC = achieving_calibration_with_witness(features, labels, classifier)

    data_class.append((base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
    kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce_Emp_AL_RF, \
    isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC))

# save data_class
save_df = pd.DataFrame(data_class)
filename = prefix + 'all.csv'
save_df.to_csv(filename)

In [ ]:


fig, ax = plt.subplots(nrows=2, ncols=4)

plt.rcParams["font.family"] = "monospace"
plt.rcParams['font.size']=15

fig.set_size_inches((12, 6))

loc_data = [[(-0.01, 0.005), (0.007, 0.002), (0, 0.001), (.025, -.01), (0, -0.02), (0, .001), (0, 0.005)], \
            [(0, -.02), (.15, .005), (-.02, .003), (.15, .01), (.1, -.02), (.21, -.001), (.3, -.002)], \
            [(-.05, .003), (.06, .005), (0, .002), (-.04, .01), (0, -.02), (.09, .005), (.13, -.005)], \
            [(.0015, -.02), (0, .005), (0, .002), (.005, -.013), (-.002, -.02), (.005, .000), (-.002, .003)]]


loc_data_2 = [[(0, .003), (0.0, 0.002), (0, .001), (0.015, -.015), (0, -.025), (0.0, 0.002), (0, 0.0015)], \
            [(0, -0.02), (0.015, -0.02), (-.005, 0.002), (0.0, 0.003), (0, 0.002), (.05, -.0024), (.06, -0.013)],\
            [(0, 0.002), (0.005, 0.0015), (0.008, 0.002), (0.01, -0.017), (0, 0), (0.01, .003), (0, 0.002)], \
            [(.001, 0.0025), (-0.001, 0.0025), (0, 0.002), (-.0015, -.012), (0, -.02), (-0.0013, 0.001), (0.002, 0)]]

i = 0

for col in ax[0].reshape(-1):
  classifier = base_classifiers[i]
  base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
  kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, \
  isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC  = data_class[i]

  x = np.array([np.mean(base_KCE, axis = 0), np.mean(kmulcal_KCE, axis = 0), np.mean(lsboost_KCE, axis = 0), \
                np.mean(kmulcal_binned_KCE, axis = 0), np.mean(MCBoost_KCE, axis = 0), np.mean(isotonic_KCE, axis = 0), np.mean(ablated_KCE, axis = 0)])
  y = np.array([np.mean(base_msce, axis = 0), np.mean(kmulcal_msce, axis = 0), np.mean(lsboost_msce, axis = 0), \
                np.mean(kmulcal_msce_binned, axis = 0), np.mean(MCBoost_msce, axis=0), np.mean(isotonic_msce, axis = 0), np.mean(ablated_msce, axis = 0)])
  z = np.mean(AUC, axis = 0)
  x_std = np.array([np.std(base_KCE, axis = 0), np.std(kmulcal_KCE, axis = 0), np.std(lsboost_KCE, axis = 0), np.std(kmulcal_binned_KCE, axis = 0), np.std(MCBoost_KCE, axis = 0), np.std(isotonic_KCE, axis = 0), np.std(ablated_KCE, axis = 0)])
  y_std = np.array([np.std(base_msce, axis = 0), np.std(kmulcal_msce, axis = 0), np.std(lsboost_msce, axis = 0),  np.std(kmulcal_msce_binned, axis = 0), np.std(MCBoost_msce, axis=0), np.std(isotonic_msce, axis = 0), np.std(ablated_msce, axis = 0)])


  colors = ['steelblue','darkgoldenrod','coral','gray','limegreen', 'darkred', 'indigo']
  markersize = 40
  sns.set(style="whitegrid", color_codes=True)
  base = col.scatter(x[0], y[0], c=colors[0], marker = 'o', s = markersize)
  kma = col.scatter(x[1], y[1], c=colors[1], marker = 's', s = markersize)
  lsb = col.scatter(x[2], y[2], c=colors[2], marker = 'p', s = markersize)
  kmak = col.scatter(x[3], y[3], c=colors[3], marker = 'x', s = markersize)
  mcb = col.scatter(x[4], y[4], c=colors[4], marker = '^', s = markersize)
  iso = col.scatter(x[5], y[5], c=colors[5], marker = 'd', s = markersize)
  abl = col.scatter(x[6], y[6], c=colors[6], marker = 'h', s = markersize)

  col.errorbar(x[0], y[0], xerr=x_std[0], yerr=y_std[0], linestyle='', color = colors[0], alpha = 1)
  col.errorbar(x[1], y[1], xerr=x_std[1], yerr=y_std[1], linestyle='', color= colors[1], alpha = 1)
  col.errorbar(x[2], y[2], xerr=x_std[2], yerr=y_std[2], linestyle='', color= colors[2], alpha = 1)
  col.errorbar(x[3], y[3], xerr=x_std[3], yerr=y_std[3], linestyle='', color= colors[3], alpha = 1)
  col.errorbar(x[4], y[4], xerr=x_std[4], yerr=y_std[4], linestyle='', color= colors[4], alpha = 1)
  col.errorbar(x[5], y[5], xerr=x_std[5], yerr=y_std[5], linestyle='', color= colors[5], alpha = 1)
  col.errorbar(x[6], y[6], xerr=x_std[6], yerr=y_std[6], linestyle='', color= colors[6], alpha = 1)

  col.set_xlabel('KME', weight='bold')
  col.set_ylabel('MSCE', weight='bold')


  cl = classifier.split("_")
  if len(cl) == 1:
    cl = cl[0]
  else:
    cl = cl[0] + " " + cl[1]

  col.set_title(cl, weight='bold')
  col.tick_params(axis='both', labelsize=9)
  col.set_ylim(bottom=-.01, top=.2)
  col.set_xlim(left=0)


  l = [f'{score:.3f}' for score in z]  # Label AUC for each point
  for j, label in enumerate(l):
    col.text(x[j] + loc_data[i][j][0], y[j] + loc_data[i][j][1], label, fontsize=9, ha='center', va='bottom', c = colors[j])

  i += 1

i = 0

loc_data = loc_data_2

for col in ax[1].reshape(-1):
  classifier = base_classifiers[i]
  base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
  kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, \
  isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC  = data2[i]

  x = np.array([np.mean(base_KCE, axis = 0), np.mean(kmulcal_KCE, axis = 0), np.mean(lsboost_KCE, axis = 0), \
                np.mean(kmulcal_binned_KCE, axis = 0), np.mean(MCBoost_KCE, axis = 0), np.mean(isotonic_KCE, axis = 0), np.mean(ablated_KCE, axis = 0)])
  y = np.array([np.mean(base_msce, axis = 0), np.mean(kmulcal_msce, axis = 0), np.mean(lsboost_msce, axis = 0), \
                np.mean(kmulcal_msce_binned, axis = 0), np.mean(MCBoost_msce, axis=0), np.mean(isotonic_msce, axis = 0), np.mean(ablated_msce, axis = 0)])
  z = np.mean(AUC, axis = 0)
  x_std = np.array([np.std(base_KCE, axis = 0), np.std(kmulcal_KCE, axis = 0), np.std(lsboost_KCE, axis = 0), np.std(kmulcal_binned_KCE, axis = 0), np.std(MCBoost_KCE, axis = 0), np.std(isotonic_KCE, axis = 0), np.std(ablated_KCE, axis = 0)])
  y_std = np.array([np.std(base_msce, axis = 0), np.std(kmulcal_msce, axis = 0), np.std(lsboost_msce, axis = 0),  np.std(kmulcal_msce_binned, axis = 0), np.std(MCBoost_msce, axis=0), np.std(isotonic_msce, axis = 0), np.std(ablated_msce, axis = 0)])


  colors = ['steelblue','darkgoldenrod','coral','gray','limegreen', 'darkred', 'indigo']
  markersize = 40
  sns.set(style="whitegrid", color_codes=True)
  base = col.scatter(x[0], y[0], c=colors[0], marker = 'o', s = markersize)
  kma = col.scatter(x[1], y[1], c=colors[1], marker = 's', s = markersize)
  lsb = col.scatter(x[2], y[2], c=colors[2], marker = 'p', s = markersize)
  kmak = col.scatter(x[3], y[3], c=colors[3], marker = 'x', s = markersize)
  mcb = col.scatter(x[4], y[4], c=colors[4], marker = '^', s = markersize)
  iso = col.scatter(x[5], y[5], c=colors[5], marker = 'd', s = markersize)
  abl = col.scatter(x[6], y[6], c=colors[6], marker = 'h', s = markersize)

  col.errorbar(x[0], y[0], xerr=x_std[0], yerr=y_std[0], linestyle='', color = colors[0], alpha = 1)
  col.errorbar(x[1], y[1], xerr=x_std[1], yerr=y_std[1], linestyle='', color= colors[1], alpha = 1)
  col.errorbar(x[2], y[2], xerr=x_std[2], yerr=y_std[2], linestyle='', color= colors[2], alpha = 1)
  col.errorbar(x[3], y[3], xerr=x_std[3], yerr=y_std[3], linestyle='', color= colors[3], alpha = 1)
  col.errorbar(x[4], y[4], xerr=x_std[4], yerr=y_std[4], linestyle='', color= colors[4], alpha = 1)
  col.errorbar(x[5], y[5], xerr=x_std[5], yerr=y_std[5], linestyle='', color= colors[5], alpha = 1)
  col.errorbar(x[6], y[6], xerr=x_std[6], yerr=y_std[6], linestyle='', color= colors[6], alpha = 1)

  col.set_xlabel('KME', weight='bold')
  col.set_ylabel('MSCE', weight='bold')


  cl = classifier.split("_")
  if len(cl) == 1:
    cl = cl[0]
  else:
    cl = cl[0] + " " + cl[1]

  col.set_title(cl, weight='bold')
  col.tick_params(axis='both', labelsize=9)
  col.set_ylim(bottom=-.01, top=.2)
  col.set_xlim(left=0)


  l = [f'{score:.3f}' for score in z]  # Label AUC for each point
  for j, label in enumerate(l):
    col.text(x[j] + loc_data[i][j][0], y[j] + loc_data[i][j][1], label, fontsize=9, ha='center', va='bottom', c = colors[j])

  i += 1
"""
for col in ax[1].reshape(-1):
  col.axis("off")"""

fig.tight_layout()
#fig.legend((base, kma, lsb, kmak, mcb, iso, abl), (f'baseline', f'KMAcc', f'LS Boost', f'KMAcc + KMeans', f'MC Boost', f'KMAcc + Isotonic Calibration', f'Baseline + Isotonic Calibration'), loc='outside lower center', ncols = 4)
plt.show()


In [ ]:



fig, ax = plt.subplots(nrows=1, ncols=4)

plt.rcParams["font.family"] = "monospace"
plt.rcParams['font.size']=15

fig.set_size_inches((12, 3))
fig.tight_layout()
loc_data = [[(-0.01, 0.005), (0.007, 0.002), (0, 0.001), (.025, -.01), (0, -0.02), (0, .001), (0, 0.005)], \
            [(0, -.02), (.15, .005), (-.02, .003), (.15, .01), (.1, -.02), (.21, -.001), (.3, -.002)], \
            [(-.05, .003), (.06, .005), (0, .002), (-.04, .01), (0, -.02), (.09, .005), (.13, -.005)], \
            [(.0015, -.02), (0, .005), (0, .002), (.005, -.013), (-.002, -.02), (.005, .000), (-.002, .003)]]

i = 0

for col in ax.reshape(-1):
  classifier = base_classifiers[i]
  base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
  kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, \
  isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC  = data_class[i]

  x = np.array([np.mean(base_KCE, axis = 0), np.mean(kmulcal_KCE, axis = 0), np.mean(lsboost_KCE, axis = 0), \
                np.mean(kmulcal_binned_KCE, axis = 0), np.mean(MCBoost_KCE, axis = 0), np.mean(isotonic_KCE, axis = 0), np.mean(ablated_KCE, axis = 0)])
  y = np.array([np.mean(base_msce, axis = 0), np.mean(kmulcal_msce, axis = 0), np.mean(lsboost_msce, axis = 0), \
                np.mean(kmulcal_msce_binned, axis = 0), np.mean(MCBoost_msce, axis=0), np.mean(isotonic_msce, axis = 0), np.mean(ablated_msce, axis = 0)])
  z = np.mean(AUC, axis = 0)
  x_std = np.array([np.std(base_KCE, axis = 0), np.std(kmulcal_KCE, axis = 0), np.std(lsboost_KCE, axis = 0), np.std(kmulcal_binned_KCE, axis = 0), np.std(MCBoost_KCE, axis = 0), np.std(isotonic_KCE, axis = 0), np.std(ablated_KCE, axis = 0)])
  y_std = np.array([np.std(base_msce, axis = 0), np.std(kmulcal_msce, axis = 0), np.std(lsboost_msce, axis = 0),  np.std(kmulcal_msce_binned, axis = 0), np.std(MCBoost_msce, axis=0), np.std(isotonic_msce, axis = 0), np.std(ablated_msce, axis = 0)])


  colors = ['steelblue','darkgoldenrod','coral','gray','limegreen', 'darkred', 'indigo']
  markersize = 40
  sns.set(style="whitegrid", color_codes=True)
  base = col.scatter(x[0], y[0], c=colors[0], marker = 'o', s = markersize)
  kma = col.scatter(x[1], y[1], c=colors[1], marker = 's', s = markersize)
  lsb = col.scatter(x[2], y[2], c=colors[2], marker = 'p', s = markersize)
  kmak = col.scatter(x[3], y[3], c=colors[3], marker = 'x', s = markersize)
  mcb = col.scatter(x[4], y[4], c=colors[4], marker = '^', s = markersize)
  iso = col.scatter(x[5], y[5], c=colors[5], marker = 'd', s = markersize)
  abl = col.scatter(x[6], y[6], c=colors[6], marker = 'h', s = markersize)

  col.errorbar(x[0], y[0], xerr=x_std[0], yerr=y_std[0], linestyle='', color = colors[0], alpha = 1)
  col.errorbar(x[1], y[1], xerr=x_std[1], yerr=y_std[1], linestyle='', color= colors[1], alpha = 1)
  col.errorbar(x[2], y[2], xerr=x_std[2], yerr=y_std[2], linestyle='', color= colors[2], alpha = 1)
  col.errorbar(x[3], y[3], xerr=x_std[3], yerr=y_std[3], linestyle='', color= colors[3], alpha = 1)
  col.errorbar(x[4], y[4], xerr=x_std[4], yerr=y_std[4], linestyle='', color= colors[4], alpha = 1)
  col.errorbar(x[5], y[5], xerr=x_std[5], yerr=y_std[5], linestyle='', color= colors[5], alpha = 1)
  col.errorbar(x[6], y[6], xerr=x_std[6], yerr=y_std[6], linestyle='', color= colors[6], alpha = 1)

  col.set_xlabel('KME', weight='bold')
  col.set_ylabel('MSCE', weight='bold')


  cl = classifier.split("_")
  if len(cl) == 1:
    cl = cl[0]
  else:
    cl = cl[0] + " " + cl[1]

  col.set_title(cl, weight='bold')
  col.tick_params(axis='both', labelsize=9)
  col.set_ylim(bottom=-.01, top=.2)
  col.set_xlim(left=0)


  l = [f'{score:.3f}' for score in z]  # Label AUC for each point
  for j, label in enumerate(l):
    col.text(x[j] + loc_data[i][j][0], y[j] + loc_data[i][j][1], label, fontsize=9, ha='center', va='bottom')

  i += 1
#fig.legend((base, kma, lsb, kmak, mcb, iso, abl), (f'baseline', f'KMultiAcc', f'LS Boost', f'KMultiAcc + KMeans', f'MC Boost', f'KMultiAcc + Isotonic Calibration', f'Baseline + Isotonic Calibration'), loc='outside lower center', ncols = 4)
fig.tight_layout()
plt.show()


In [ ]:
# Health Task WI
# Health Public Coverage Task WI
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states=["WI"], download=True)
features, labels, _ = ACSPublicCoverage.df_to_numpy(data)
prefix = "Health_WI_"

#plot_for_task(features, labels, base_classifiers, prefix)

base_classifiers = ["Logistic_Regression", "Naive_Bayes", "Random_Forest", "NN"]



data2 = []

for classifier in base_classifiers:
    base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
    kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce_Emp_AL_RF, \
    isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC = achieving_calibration_with_witness(features, labels, classifier)

    data2.append((base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
    kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce_Emp_AL_RF, \
    isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC))

# save data_class
save_df = pd.DataFrame(data2)
filename = prefix + 'all.csv'
save_df.to_csv(filename)

In [ ]:


fig, ax = plt.subplots(nrows=1, ncols=4)

plt.rcParams["font.family"] = "monospace"
plt.rcParams['font.size']=15

fig.set_size_inches((12, 3))

loc_data = [[(0, .003), (0.0, 0.002), (0, .001), (0.015, -.015), (0, -.025), (0.0, 0.002), (0, 0.0015)], \
            [(0, -0.02), (0.015, -0.02), (-.005, 0.002), (0.0, 0.003), (0, 0.002), (.05, -.0024), (.06, -0.013)],\
            [(0, 0.002), (0.005, 0.0015), (0.008, 0.002), (0.01, -0.017), (0, 0), (0.01, .003), (0, 0.002)], \
            [(.001, 0.0025), (-0.001, 0.0025), (0, 0.002), (-.0015, -.012), (0, -.02), (-0.0013, 0.001), (0.002, 0)]]

i = 0


for col in ax.reshape(-1):
  classifier = base_classifiers[i]
  base_msce, base_KCE, kmulcal_msce, kmulcal_KCE, lsboost_msce, lsboost_KCE, base_msce_binned, base_binned_KCE, \
  kmulcal_msce_binned, kmulcal_binned_KCE, MCBoost_msce, MCBoost_KCE, isotonic_msce, \
  isotonic_KCE, sigmoid_msce, sigmoid_KCE, ablated_msce, ablated_KCE, AUC  = data2[i]

  x = np.array([np.mean(base_KCE, axis = 0), np.mean(kmulcal_KCE, axis = 0), np.mean(lsboost_KCE, axis = 0), \
                np.mean(kmulcal_binned_KCE, axis = 0), np.mean(MCBoost_KCE, axis = 0), np.mean(isotonic_KCE, axis = 0), np.mean(ablated_KCE, axis = 0)])
  y = np.array([np.mean(base_msce, axis = 0), np.mean(kmulcal_msce, axis = 0), np.mean(lsboost_msce, axis = 0), \
                np.mean(kmulcal_msce_binned, axis = 0), np.mean(MCBoost_msce, axis=0), np.mean(isotonic_msce, axis = 0), np.mean(ablated_msce, axis = 0)])
  z = np.mean(AUC, axis = 0)
  x_std = np.array([np.std(base_KCE, axis = 0), np.std(kmulcal_KCE, axis = 0), np.std(lsboost_KCE, axis = 0), np.std(kmulcal_binned_KCE, axis = 0), np.std(MCBoost_KCE, axis = 0), np.std(isotonic_KCE, axis = 0), np.std(ablated_KCE, axis = 0)])
  y_std = np.array([np.std(base_msce, axis = 0), np.std(kmulcal_msce, axis = 0), np.std(lsboost_msce, axis = 0),  np.std(kmulcal_msce_binned, axis = 0), np.std(MCBoost_msce, axis=0), np.std(isotonic_msce, axis = 0), np.std(ablated_msce, axis = 0)])


  colors = ['steelblue','darkgoldenrod','coral','gray','limegreen', 'darkred', 'indigo']
  markersize = 40
  sns.set(style="whitegrid", color_codes=True)
  base = col.scatter(x[0], y[0], c=colors[0], marker = 'o', s = markersize)
  kma = col.scatter(x[1], y[1], c=colors[1], marker = 's', s = markersize)
  lsb = col.scatter(x[2], y[2], c=colors[2], marker = 'p', s = markersize)
  kmak = col.scatter(x[3], y[3], c=colors[3], marker = 'x', s = markersize)
  mcb = col.scatter(x[4], y[4], c=colors[4], marker = '^', s = markersize)
  iso = col.scatter(x[5], y[5], c=colors[5], marker = 'd', s = markersize)
  abl = col.scatter(x[6], y[6], c=colors[6], marker = 'h', s = markersize)

  col.errorbar(x[0], y[0], xerr=x_std[0], yerr=y_std[0], linestyle='', color = colors[0], alpha = 1)
  col.errorbar(x[1], y[1], xerr=x_std[1], yerr=y_std[1], linestyle='', color= colors[1], alpha = 1)
  col.errorbar(x[2], y[2], xerr=x_std[2], yerr=y_std[2], linestyle='', color= colors[2], alpha = 1)
  col.errorbar(x[3], y[3], xerr=x_std[3], yerr=y_std[3], linestyle='', color= colors[3], alpha = 1)
  col.errorbar(x[4], y[4], xerr=x_std[4], yerr=y_std[4], linestyle='', color= colors[4], alpha = 1)
  col.errorbar(x[5], y[5], xerr=x_std[5], yerr=y_std[5], linestyle='', color= colors[5], alpha = 1)
  col.errorbar(x[6], y[6], xerr=x_std[6], yerr=y_std[6], linestyle='', color= colors[6], alpha = 1)

  col.set_xlabel('KME', weight='bold')
  col.set_ylabel('MSCE', weight='bold')


  cl = classifier.split("_")
  if len(cl) == 1:
    cl = cl[0]
  else:
    cl = cl[0] + " " + cl[1]

  col.set_title(cl, weight='bold')
  col.tick_params(axis='both', labelsize=9)
  col.set_ylim(bottom=-.01, top=.2)
  col.set_xlim(left=0)


  l = [f'{score:.3f}' for score in z]  # Label AUC for each point
  for j, label in enumerate(l):
    col.text(x[j] + loc_data[i][j][0], y[j] + loc_data[i][j][1], label, fontsize=9, ha='center', va='bottom')

  i += 1

fig.tight_layout()
#fig.legend((base, kma, lsb, kmak, mcb, iso, abl), (f'baseline', f'KMAcc', f'LS Boost', f'KMAcc + KMeans', f'MC Boost', f'KMAcc + Isotonic Calibration', f'Baseline + Isotonic Calibration'), loc='outside lower center', ncols = 4)
plt.show()
